# Experiment 02 — Output Reuse (OR) under DQN · SMOKE FIRST

Paper block 1 (post-PQC inference). Sweep R ∈ {4, 16} at **1 seed** to decide whether the
full sweep is worth the hours. Paper (PPO): OR helps hybrid agents but not classical ones (a genuine quantum interaction). Question: does it
transfer to DQN?

**Why smoke first.** exp01 showed the hybrid barely learns under DQN in this
regime (greedy ~44, high variance). An OR effect on a weakly-learning agent may
vanish into noise — the trap FIX-01 fell into twice. This notebook runs the two
extreme R values at one seed: if they visibly separate, the full sweep is worth
it; if not, OR is not measurable here and that itself is the finding.

Hybrid runs are ~1–4h **per cell**. Smoke = 2 cells. Full sweep (section 4) is
8 cells at 3 seeds — only launch it if smoke says go.

---
## 1. Environment

In [ ]:
from google.colab import userdata
GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    GH_TOKEN = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if GH_TOKEN else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import sys, subprocess, pathlib, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp02")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd()/"results"/"exp02"; CODE = pathlib.Path.cwd()
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists(): subprocess.run(["git","-C",str(CODE),"pull","--quiet"], check=False)
    else: subprocess.run(["git","clone","--quiet","-b",BRANCH,REPO_URL,str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(CODE/"requirements.txt")], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(CODE)], check=True)
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","jax","jaxlib"], check=False)

rev = subprocess.run(["git","-C",str(CODE),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip()
print("code:", CODE, "@", rev, " results:", RESULTS)

import sys as _sys
_need=False
if "autoray.autoray" in _sys.modules:
    import autoray.autoray as _aa; _need = not hasattr(_aa,"NumpyMimic")
if "jax" in _sys.modules: _need=True
if _need:
    print("Incompatible modules loaded -> restarting."); os.kill(os.getpid(), 9)
else:
    print("Clean environment: no restart needed. Continue below.")

### After restarting (if it did), run from here

In [ ]:
import sys, pathlib, time, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp02")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd()/"results"/"exp02"; CODE = pathlib.Path.cwd()
sys.path.insert(0, str(CODE/"src"))

import qrl_dissection
from qrl_dissection import analysis
from qrl_dissection.core.configs import hybrid_or_config, OR_REPEATS, hybrid_or_config
from qrl_dissection.dqn import SafeDQN, GreedyEvalConfig
import pandas as pd, numpy as np
print("ready. sweep points:", OR_REPEATS)

---
## 2. Throughput probe

One 1500-step hybrid run to size the cost before committing.

In [ ]:
from qrl_dissection.dqn import RunSpec, run_arm
t0=time.time()
_=run_arm(RunSpec(arm="hybrid_fig4", seed=99, fix_autoreset=True, total_timesteps=1500, tag="probe"),
          outdir=RESULTS/"_probe")
dt=time.time()-t0; sps=1500/dt
print(f"{sps:.1f} steps/s -> 60k ~ {60_000/sps/60:.1f} min/cell")
print(f"smoke (2 cells) ~ {2*60_000/sps/60:.0f} min | full sweep ~ {8*60_000/sps/60:.0f} min")

---
## 3. SMOKE — two extreme R values, one seed

Runs R=4 and R=16 at seed 1. Resumable.

In [ ]:
SMOKE_R = [4, 16]
STEPS = 100_000   # match exp03/DR for cross-block comparability
KW = dict(batch_size=128, buffer_size=10_000, train_frequency=10)

def run_cell(val, seed):
    name = f"hybrid_OR{val}__s{seed}"
    mp = RESULTS/f"{name}.manifest.json"
    if mp.exists():
        print(f"[skip] {name}"); return json.loads(mp.read_text())
    cfg = hybrid_or_config(val)
    print(f"[run ] {name}", flush=True)
    r = SafeDQN(agent_type="hybrid", agent_config=cfg, run_name=name, seed=seed,
                fix_autoreset=True, eval_cfg=GreedyEvalConfig(every_steps=10_000),
                outdir=RESULTS, **KW)
    out = r.train(STEPS, progress_bar=True)
    m = {"name": name, "reuse_repetitions": val, "seed": seed, "outcome": out.__dict__}
    mp.write_text(json.dumps(m, indent=2, default=str))
    print(f"       ok {out.wall_seconds}s greedy_best from eval csv")
    return m

for v in SMOKE_R:
    run_cell(v, 1)

---
## 4. Smoke verdict — separate or not?

In [ ]:
def summarise(prefix_dir):
    rows=[]
    for mp in sorted(pathlib.Path(prefix_dir).glob("hybrid_OR*.manifest.json")):
        m=json.loads(mp.read_text()); oc=m["outcome"]
        rew,step=analysis.load_episodes(oc["episodes_csv"])
        best=float(pd.Series(rew).rolling(50).mean().max())
        gb=np.nan
        if oc.get("eval_csv") and pathlib.Path(oc["eval_csv"]).exists():
            _,sc=analysis.load_eval(oc["eval_csv"]); gb=float(np.max(sc)) if len(sc) else np.nan
        val=int(m["name"].split("OR")[1].split("__")[0])
        rows.append(dict(**{"R":val}, seed=m["seed"], best_ma50=round(best,1), greedy_best=round(gb,1)))
    return pd.DataFrame(rows).sort_values(["R","seed"])

df=summarise(RESULTS)
display(df)
lo, hi = 4, 16
try:
    g_lo=df[df["R"]==lo].greedy_best.mean(); g_hi=df[df["R"]==hi].greedy_best.mean()
    print(f"\ngreedy_best: R={lo} -> {g_lo:.1f} | R={hi} -> {g_hi:.1f} | gap {g_hi-g_lo:+.1f}")
    print("\nVERDICT:")
    if abs(g_hi-g_lo) >= 15:
        print(f"  clear separation -> the full R sweep is worth it. Run section 5.")
    else:
        print(f"  no clear separation at 1 seed. OR may be unmeasurable in this weak-learning")
        print(f"  DQN regime. Consider: (a) report that as the finding, or (b) first get")
        print(f"  the base hybrid to learn better (more steps / another env) before sweeping.")
except Exception as e:
    print("verdict skipped:", e)

---
## 5. FULL sweep — only if smoke said go

The whole OR_REPEATS at 3 seeds. Or just run the CLI from a terminal cell:
`!python {CODE}/experiments/exp02_dqn_cartpole_output_reuse.py --outdir {RESULTS}`

In [ ]:
FULL = [4, 8, 16, 32]
SEEDS = [1, 2, 3]

# guard: don't auto-run the expensive sweep. Set RUN_FULL=True deliberately.
RUN_FULL = False
if RUN_FULL:
    for v in FULL:
        for s in SEEDS:
            run_cell(v, s)
    display(summarise(RESULTS))
else:
    print("RUN_FULL is False. Set it True and re-run this cell to launch the full sweep")
    print("(only after the smoke verdict in section 4 says the values separate).")

---
## 6. Recording

Paste the section-4 (and section-5 if run) table into `docs/RESULTS-LOG.md` under
*Experiment 02*, with git rev and `verify_env` output. Commit.

**Remember plan B** (standing note in `docs/ROADMAP.md`): whatever the 3-seed
sweep shows, it is COVERAGE. Before writing it up as a conclusion, re-run at
8–10 seeds. Do not let these 3-seed numbers become the final numbers.